<a href="https://colab.research.google.com/github/ashuyad/learning-transformers/blob/main/notebooks/fine_tune_translation_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("ArmelRandy/kde4")

README.md:   0%|          | 0.00/464 [00:00<?, ?B/s]

data/train-00000-of-00001-e6e984806935f4(…): reconstructing file:   0%|          |  0.00B / 5.30MB            

data/train-00000-of-00001-e6e984806935f4(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/20058 [00:00<?, ? examples/s]

In [ ]:
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['en', 'fr'],
        num_rows: 20058
    })
})

In [ ]:
split_datasets = raw_dataset["train"].train_test_split(train_size=0.9, seed=20)
split_datasets

DatasetDict({
    train: Dataset({
        features: ['en', 'fr'],
        num_rows: 18052
    })
    test: Dataset({
        features: ['en', 'fr'],
        num_rows: 2006
    })
})

In [ ]:
split_datasets["validation"] = split_datasets.pop("test")

In [ ]:
split_datasets["train"][1]

{'en': 'The PERMUT() function returns the number of permutations. The first parameter is the number of elements, and the second parameter is the number of elements used in the permutation.',
 'fr': "La fonction PERMUT() renvoie le nombre de permutations. Le premier paramètre est le nombre d'éléments et le second est le nombre d'éléments à permuter."}

In [ ]:
from transformers import AutoTokenizer

model_checkpoint = "Helsinki-NLP/opus-mt-en-fr"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, return_tensor="pt")

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/778k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [ ]:
max_length = 128

def preprocess_function(examples):
    inputs = examples["en"]
    targets = examples["fr"]
    model_inputs = tokenizer(
        inputs, text_target=targets, max_length=max_length, truncation=True
    )
    return model_inputs

In [ ]:
tokenized_dataset = split_datasets.map(preprocess_function, batched=True, remove_columns= split_datasets["train"].column_names)

Map:   0%|          | 0/18052 [00:00<?, ? examples/s]

Map:   0%|          | 0/2006 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained(model_checkpoint)

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  301MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  301MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

In [ ]:
!pip install evaluate sacrebleu
import evaluate

metric = evaluate.load("sacrebleu")

In [ ]:
import numpy as np

def compute_metrics(eval_preds):
  preds, labels = eval_preds

  if isinstance(preds, tuple):
    preds = preds[0]

  decoded_preds = tokenizer.batch_decode(preds, skip_special_token=True)

  labels = np.where(labels!= -100, labels, tokenizer.pad_token_id)
  decoded_labels = tokenizer.batch_decode(labels, skip_special_token=True)

  decoded_preds = [pred.strip() for pred in decoded_preds]
  decoded_labels = [label.strip() for label in decoded_labels]

  result = metric.compute(predictions= decoded_preds, references= decoded_labels)

  return {"bleu": result["score"]}


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from transformers import Seq2SeqTrainingArguments

args = Seq2SeqTrainingArguments(
    f"finetuned-kde4-en-to-fr",
    eval_strategy = "no",
    save_strategy="epoch",
    learning_rate= 2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    weight_decay=0.01,
    save_total_limit=3,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    push_to_hub=True
)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    model,
    args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.evaluate(max_length=max_length)

Training Loss,Validation Loss,Step,Bleu
No log,1.156332,0,97.427410


{'eval_loss': 1.1563315391540527, 'eval_bleu': 97.42740950446277}

In [ ]:
trainer.train()

Step,Training Loss
500,1.011102
1000,0.898229
1500,0.858197


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1695, training_loss=0.9131194078113477, metrics={'train_runtime': 443.9371, 'train_samples_per_second': 121.99, 'train_steps_per_second': 3.818, 'total_flos': 1758417299177472.0, 'train_loss': 0.9131194078113477, 'epoch': 3.0})

In [ ]:
trainer.evaluate(max_length=max_length)

Training Loss,Validation Loss,Step,Bleu
0.858197,0.877754,1695,97.747111


{'eval_loss': 0.8777536153793335, 'eval_bleu': 97.7471108331219}

In [ ]:
trainer.push_to_hub(tags="translation", commit_message="Training complete")